# Loading Tabular Data

The fastest way to get music data into TimeToAlign! is through **tabular loaders**. If your data is in CSV or TSV format, you're just 3 lines of code away from analysis.

**What you'll learn:**
- Load music annotations from TSV/CSV files
- Access event counts, coordinate ranges, and metadata
- Create timelines from loaded data
- Create custom loaders with different `extra_columns` strategies
- Use `Field` for nested JSON column access

**Time:** 15 minutes

## TL;DR

```python
from timetoalign.loader.tabular import Ms3Loader

loader = Ms3Loader()
loader.load("beethoven.notes.tsv")

df = loader.events.to_pandas()       # Get as DataFrame
timeline = loader.create_timeline()  # Create Timeline
```

## Setup

In [1]:
from pathlib import Path

# Specimen directories
SPECIMENS = Path(".").resolve().parents[1] / ".." / "dashboard" / "specimens"
BEETHOVEN = SPECIMENS / "beethoven_woo71"
THORESEN = SPECIMENS / "thoresen"

# Available files
{
    "Beethoven files": [f.name for f in BEETHOVEN.glob("WoO71.*.tsv")],
    "Thoresen files": [f.name for f in THORESEN.glob("*.tsv")],
}

{'Beethoven files': ['WoO71.chords.tsv',
  'WoO71.measures.tsv',
  'WoO71.notes.tsv'],
 'Thoresen files': ['thoresen_test_h.tsv', 'thoresen_test.tsv']}

## Loading Notes from TSV

The `Ms3Loader` handles TSV files exported from the [ms3](https://github.com/johentsch/ms3) parser, which processes MuseScore files.

**Three lines of code:**

In [2]:
from timetoalign.loader.tabular import Ms3Loader

loader = Ms3Loader()
loader.load(BEETHOVEN / "WoO71.notes.tsv")

f"{len(loader.events):,} notes loaded"

'4,753 notes loaded'

## Converting to pandas

Use `to_pandas()` to get a DataFrame with clean coordinate values:

In [3]:
loader.events.to_pandas().head()

,id,name,temporal_type,event_type,start,end,duration,octave,midi,voice,tpc,mn,chord_id,mc,staff
0,e000000,A3,interval,Note,0,1/4,1/4,3,57,1,3,0,3,1,2
1,e000001,E4,interval,Note,0,1/4,1/4,4,64,2,4,0,2,1,1
2,e000002,A4,interval,Note,0,1/8,1/8,4,69,1,3,0,0,1,1
3,e000003,C#5,interval,Note,0,1/8,1/8,5,73,1,7,0,0,1,1
4,e000004,E5,interval,Note,1/2,5/8,1/8,5,76,1,4,0,1,1,1


## Quick Statistics

The loader provides immediate access to summary information:

In [4]:
{
    "event_count": len(loader.events),
    "coordinate_range": loader.events.coordinate_range(),
    "unit": str(loader.unit),
    "number_type": str(loader.number_type),
}

{'event_count': 4753,
 'coordinate_range': (0.0, 877.75),
 'unit': 'quarters',
 'number_type': 'fraction'}

## Creating Timelines

TimeToAlign! represents temporal data as **Timelines**:

In [ ]:
timeline = loader.create_timeline(uid="beethoven_notes")
timeline

## Custom Loaders for Non-Standard Formats

For files that don't match the ms3 format, create a custom loader by subclassing `TsvLoader` or `CsvLoader`.

Let's load the Thoresen annotations file which has a different column structure:

In [6]:
import pandas as pd

pd.read_csv(THORESEN / "thoresen_test.tsv", sep="\t", nrows=3)

,event_id,alignment_group_id,start_time_sec,duration_sec,event_type,graphical_element_id,image_filename,rect_coords_json,text_content,text_anchor_xy_json,layer_order,description
0,annot_cue_001,NaN,0.0,5.0,rectangle,rect_a,thoresen_2010_form-building-patterns_p90-91_pa...,"{""x"": 10, ""y"": 90, ""width"": 148, ""height"": 55}",NaN,NaN,NaN,NaN
1,annot_cue_002,NaN,1.5,4.0,rectangle,rect_b,thoresen_2010_form-building-patterns_p90-91_pa...,"{""x"": 40, ""y"": 37, ""width"": 127, ""height"": 21}",NaN,NaN,NaN,NaN
2,annot_cue_003,NaN,3.5,2.0,rectangle,rect_c,thoresen_2010_form-building-patterns_p90-91_pa...,"{""x"": 111, ""y"": 60, ""width"": 57, ""height"": 23}",NaN,NaN,NaN,NaN


### Strategy 1: Simplest Case (No Extra Columns)

The simplest custom loader just maps the core coordinate columns. No `extra_columns` means only the base event fields are loaded:

In [7]:
from timetoalign.loader.tabular import TsvLoader
from timetoalign.core import TimeUnit, NumberType

class ThoresenMinimalLoader(TsvLoader):
    """Minimal loader - only core event fields."""
    
    id_column = "event_id"
    start_column = "start_time_sec"
    duration_column = "duration_sec"
    event_type_column = "event_type"
    name_column = "description"
    
    _default_unit = TimeUnit.seconds
    coordinate_type = NumberType.float

minimal = ThoresenMinimalLoader()
minimal.load(THORESEN / "thoresen_test.tsv")
minimal.events.to_pandas()

,id,name,temporal_type,event_type,start,end,duration
0,annot_cue_001,NaN,interval,rectangle,0.0,5.00,5.00
1,annot_cue_002,NaN,interval,rectangle,1.5,5.50,4.00
2,annot_cue_003,NaN,interval,rectangle,3.5,5.50,2.00
3,annot_cue_004,NaN,interval,rectangle,34.6,39.80,5.20
4,annot_cue_005,NaN,interval,rectangle,43.5,48.00,4.50
5,annot_cue_006,NaN,interval,rectangle,71.0,75.75,4.75
6,annot_cue_007,NaN,interval,rectangle,76.0,83.50,7.50
7,annot_cue_008,NaN,interval,rectangle,90.5,94.50,4.00
8,annot_cue_009,NaN,interval,rectangle,113.4,116.40,3.00
9,annot_cue_010,NaN,interval,rectangle,121.0,128.50,7.50


### Strategy 2: Auto-Infer All Columns

Set `extra_columns = True` to automatically include all remaining columns with inferred types:

In [8]:
class ThoresenAutoLoader(TsvLoader):
    """Auto-infer all remaining columns."""
    
    id_column = "event_id"
    start_column = "start_time_sec"
    duration_column = "duration_sec"
    event_type_column = "event_type"
    name_column = "description"
    
    _default_unit = TimeUnit.seconds
    coordinate_type = NumberType.float
    
    # Include ALL remaining columns with inferred types
    extra_columns = True

auto = ThoresenAutoLoader()
auto.load(THORESEN / "thoresen_test.tsv")
auto.events.to_pandas()

,id,name,temporal_type,event_type,start,end,duration,image_filename,graphical_element_id,text_anchor_xy_json,rect_coords_json,text_content,layer_order,alignment_group_id
0,annot_cue_001,NaN,interval,rectangle,0.0,5.00,5.00,thoresen_2010_form-building-patterns_p90-91_pa...,rect_a,NaN,"{""x"": 10, ""y"": 90, ""width"": 148, ""height"": 55}",NaN,NaN,NaN
1,annot_cue_002,NaN,interval,rectangle,1.5,5.50,4.00,thoresen_2010_form-building-patterns_p90-91_pa...,rect_b,NaN,"{""x"": 40, ""y"": 37, ""width"": 127, ""height"": 21}",NaN,NaN,NaN
2,annot_cue_003,NaN,interval,rectangle,3.5,5.50,2.00,thoresen_2010_form-building-patterns_p90-91_pa...,rect_c,NaN,"{""x"": 111, ""y"": 60, ""width"": 57, ""height"": 23}",NaN,NaN,NaN
3,annot_cue_004,NaN,interval,rectangle,34.6,39.80,5.20,thoresen_2010_form-building-patterns_p90-91_pa...,rect_a2,NaN,"{""x"": 145, ""y"": 90, ""width"": 160, ""height"": 58}",NaN,NaN,NaN
4,annot_cue_005,NaN,interval,rectangle,43.5,48.00,4.50,thoresen_2010_form-building-patterns_p90-91_pa...,rect_h2,NaN,"{""x"": 385, ""y"": 46, ""width"": 139, ""height"": 20}",NaN,NaN,NaN
5,annot_cue_006,NaN,interval,rectangle,71.0,75.75,4.75,thoresen_2010_form-building-patterns_p90-91_pa...,rect_d3,NaN,"{""x"": 310, ""y"": 93, ""width"": 154, ""height"": 18}",NaN,NaN,NaN
6,annot_cue_007,NaN,interval,rectangle,76.0,83.50,7.50,thoresen_2010_form-building-patterns_p90-91_pa...,rect_b3,NaN,"{""x"": 456, ""y"": 69, ""width"": 229, ""height"": 18}",NaN,NaN,NaN
7,annot_cue_008,NaN,interval,rectangle,90.5,94.50,4.00,thoresen_2010_form-building-patterns_p90-91_pa...,rect_i4,NaN,"{""x"": 14, ""y"": 115, ""width"": 127, ""height"": 31}",NaN,NaN,NaN
8,annot_cue_009,NaN,interval,rectangle,113.4,116.40,3.00,thoresen_2010_form-building-patterns_p90-91_pa...,rect_a4,NaN,"{""x"": 663, ""y"": 82, ""width"": 97, ""height"": 23}",NaN,NaN,NaN
9,annot_cue_010,NaN,interval,rectangle,121.0,128.50,7.50,thoresen_2010_form-building-patterns_p90-91_pa...,rect_i5,NaN,"{""x"": 19, ""y"": 119, ""width"": 251, ""height"": 29}",NaN,NaN,NaN


### Strategy 3: Explicit Columns with Types

Use a dict to specify exactly which columns to include and their types:

In [9]:
class ThoresenTypedLoader(TsvLoader):
    """Explicit columns with types."""
    
    id_column = "event_id"
    start_column = "start_time_sec"
    duration_column = "duration_sec"
    event_type_column = "event_type"
    name_column = "description"
    
    _default_unit = TimeUnit.seconds
    coordinate_type = NumberType.float
    
    # Explicit columns with types
    extra_columns = {
        "image_filename": str,
        "graphical_element_id": int,
    }

typed = ThoresenTypedLoader()
typed.load(THORESEN / "thoresen_test.tsv")
typed.events.to_pandas()

,id,name,temporal_type,event_type,start,end,duration,image_filename,graphical_element_id
0,annot_cue_001,NaN,interval,rectangle,0.0,5.00,5.00,thoresen_2010_form-building-patterns_p90-91_pa...,rect_a
1,annot_cue_002,NaN,interval,rectangle,1.5,5.50,4.00,thoresen_2010_form-building-patterns_p90-91_pa...,rect_b
2,annot_cue_003,NaN,interval,rectangle,3.5,5.50,2.00,thoresen_2010_form-building-patterns_p90-91_pa...,rect_c
3,annot_cue_004,NaN,interval,rectangle,34.6,39.80,5.20,thoresen_2010_form-building-patterns_p90-91_pa...,rect_a2
4,annot_cue_005,NaN,interval,rectangle,43.5,48.00,4.50,thoresen_2010_form-building-patterns_p90-91_pa...,rect_h2
5,annot_cue_006,NaN,interval,rectangle,71.0,75.75,4.75,thoresen_2010_form-building-patterns_p90-91_pa...,rect_d3
6,annot_cue_007,NaN,interval,rectangle,76.0,83.50,7.50,thoresen_2010_form-building-patterns_p90-91_pa...,rect_b3
7,annot_cue_008,NaN,interval,rectangle,90.5,94.50,4.00,thoresen_2010_form-building-patterns_p90-91_pa...,rect_i4
8,annot_cue_009,NaN,interval,rectangle,113.4,116.40,3.00,thoresen_2010_form-building-patterns_p90-91_pa...,rect_a4
9,annot_cue_010,NaN,interval,rectangle,121.0,128.50,7.50,thoresen_2010_form-building-patterns_p90-91_pa...,rect_i5


## Nested JSON Column Access with Field

The Thoresen data has a `rect_coords_json` column containing pixel coordinates as JSON:
```json
{"x": 10, "y": 90, "width": 148, "height": 55}
```

Use `Field("column", "nested_field")` to access nested fields directly. TimeToAlign! automatically parses JSON when needed:

In [10]:
from timetoalign.loader import Field, ComputedField

class ThoresenGraphicalLoader(TsvLoader):
    """Loader using PIXEL coordinates from nested JSON.
    
    Field automatically parses JSON columns when accessing nested fields.
    """
    
    # Use nested fields directly - JSON is parsed automatically
    start_column = Field("rect_coords_json", "x")
    end_column = ComputedField("end", formula="rect_coords_json.x + rect_coords_json.width")
    
    _default_unit = TimeUnit.pixels
    coordinate_type = NumberType.float
    default_event_type = "Rectangle"

graphical = ThoresenGraphicalLoader()
graphical.load(THORESEN / "thoresen_test.tsv")

{
    "unit": str(graphical.unit),
    "coordinate_range": graphical.events.coordinate_range(),
}

{'unit': 'pixels', 'coordinate_range': (10.0, 760.0)}

### Two Coordinate Systems from One File

The same TSV file can create timelines in different coordinate systems:

In [11]:
# Physical timeline (seconds)
typed.events.to_pandas()[["id", "start", "end"]]

,id,start,end
0,annot_cue_001,0.0,5.00
1,annot_cue_002,1.5,5.50
2,annot_cue_003,3.5,5.50
3,annot_cue_004,34.6,39.80
4,annot_cue_005,43.5,48.00
5,annot_cue_006,71.0,75.75
6,annot_cue_007,76.0,83.50
7,annot_cue_008,90.5,94.50
8,annot_cue_009,113.4,116.40
9,annot_cue_010,121.0,128.50


In [12]:
# Graphical timeline (pixels)
graphical.events.to_pandas()[["id", "start", "end"]]

,id,start,end
0,e000000,10.0,158.0
1,e000001,40.0,167.0
2,e000002,111.0,168.0
3,e000003,145.0,305.0
4,e000004,385.0,524.0
5,e000005,310.0,464.0
6,e000006,456.0,685.0
7,e000007,14.0,141.0
8,e000008,663.0,760.0
9,e000009,19.0,270.0


### Creating Timelines

Use `create_timeline()` to convert loaded events into a Timeline object. The `diagram()` method shows an ASCII visualization:

In [ ]:
# Create Physical Timeline (seconds)
physical_tl = typed.create_timeline(uid="thoresen_physical")
physical_tl

In [ ]:
# Create Graphical Timeline (pixels)
graphical_tl = graphical.create_timeline(uid="thoresen_graphical")
graphical_tl

**Note:** Both timelines represent the same 11 events, but in different coordinate systems:
- **Physical:** `0 - 142.5 seconds` (audio time)
- **Graphical:** `10 - 760 pixels` (image coordinates)

TimeToAlign! uses these dual representations to align graphical annotations with audio.

## Summary

### Extra Columns Strategies

| Strategy | Syntax | Use Case |
|----------|--------|----------|
| None | `extra_columns` not set | Only core event fields |
| Auto-infer | `extra_columns = True` | Include all columns, infer types |
| Explicit dict | `extra_columns = {"col": type}` | Specific columns with types |

### Key API

```python
# Load
loader = Ms3Loader()
loader.load("file.tsv")

# Access
df = loader.events.to_pandas()
timeline = loader.create_timeline()

# Custom loader with explicit columns
class MyLoader(TsvLoader):
    start_column = "onset"
    duration_column = "dur"
    extra_columns = {"pitch": int, "velocity": int}

# Nested JSON field access (auto-parses JSON)
from timetoalign.loader import Field, ComputedField

class GraphicalLoader(TsvLoader):
    start_column = Field("rect_json", "x")  # JSON parsed automatically
    end_column = ComputedField("end", formula="rect_json.x + rect_json.width")
```

> **Key Takeaway:** Tabular loaders provide a declarative way to map CSV/TSV columns to TimeToAlign! events. Use `Field` for nested JSON access - parsing happens automatically.